# 端到端 capstone：从 canonical 问题到可追溯回答

本 Notebook 把[端到端验收协议](端到端验收协议.md)落成一个可重复执行的闭环：canonical 问题 → BM25 首轮检索 → 统一上下文预算 → C6 缺口检查与补查 → 真实 `glm-4-flash` 回答 → 逐字引用与同一 `glm-4-flash` 的独立请求语义 reviewer → 事后 qrels 验收。

样本固定为 3 道 answerable 题（其中 2 道是多证据题）和 1 道 canonical unanswerable 题。证据池只含 canonical `evidence_type=quote` 原文。本页使用[检查检索结果后再继续](../6.%20处理信息缺口/检查检索结果后再继续.ipynb)的四字段反思契约接入一轮本地补查；这是信息缺口教学实验，不是 33 种方法的统一 benchmark。

## 运行边界与验收协议

- 检索、缺口判断和模型提示词只接收问题文字与实际检索的 canonical quote；评测标注在所有模型响应产生后才读取。
- 普通题首轮 `top_k=8`；“模型评估与宏/微平均”题显式使用 `top_k=1` 的窄召回场景，检验首轮缺少一部分资料时系统如何继续。该设置不是算法优劣比较，也不按标注删除证据。
- 每题首轮和补查后的完整上下文都最多 1,800 字符，按完整 evidence 顺序纳入。补查增加检索与一次反思调用，但不会增加最终上下文预算。
- 缺口模型只能根据 QUESTION+首轮 CONTEXT 判断是否足够并生成补查词。若它判断足够，就记录未触发；不强制走成功分支，不使用人工答案词或参考答案补查。
- 真实模型通过项目根 `.env` 的 `ZHIPUAI_API_KEY` 调用固定 `glm-4-flash`，共 1 次缺口反思、4 次生成、4 次独立请求语义 reviewer（仍使用同一 `glm-4-flash`）。公共 `llm_call` 固定 SDK `max_retries=0`，每次调用至多一个 HTTP 请求，依赖、API 或解析错误直接抛出。
- 原始响应立即写入 Notebook 的原始调用 MIME，最终 audit 再逐题保存生成原文、reviewer 原文和补查原文。质量负结果会保存为 `fail`；只读 checker 重新解析原文并重放本地检索，不接受手填的通过结论。

In [1]:
from pathlib import Path
import json
import sys

COURSE_CANDIDATES = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
COURSE_ROOT = next((path for path in COURSE_CANDIDATES if path.name == "C7 高级 RAG 技巧" and (path / "common").is_dir()), None)
if COURSE_ROOT is None:
    raise RuntimeError(f"当前路径不在 C7 教程目录：{Path.cwd()}")
sys.path.insert(0, str(COURSE_ROOT))

from common.dataset import (
    DATASET_ROOT,
    read_jsonl,
)
from common.eval_utils import build_bm25_chunk_search, emit_tutorial_audit
from common.nontraining_utils import RAG_LLM_MODEL, llm_call
sys.path.insert(0, str(COURSE_ROOT / "7. 评估"))
from check_capstone import (
    RAW_MIME,
    evaluate_case,
    validate_gap_reflection,
    validate_generation,
    validate_semantic_review,
)
from IPython.display import display

def persist_raw_response(stage, case_id, raw):
    # 在解析前保存，不让后续解析失败抹掉已收到的服务响应。
    display({RAW_MIME: {
        "stage": stage, "query_id": case_id, "model": RAG_LLM_MODEL,
        "api_key_source": ".env:ZHIPUAI_API_KEY", "max_retries": 0,
        "raw_response": raw,
    }}, raw=True)

INSUFFICIENT_ANSWER = "资料不足，无法根据提供的上下文回答。"

if RAG_LLM_MODEL != "glm-4-flash":
    raise RuntimeError("capstone 固定要求真实 glm-4-flash；请不要用环境变量切换模型")

CASE_IDS = (
    "model_selection_with_intro_scope",
    "cross_validation_reliability",
    "model_evaluation_and_macro_micro",
    "book_evidence_boundary",
)
GAP_CASE_ID = "model_evaluation_and_macro_micro"
TOP_K = 8
CONTEXT_BUDGET = 1800
print("model =", RAG_LLM_MODEL, "; cases =", len(CASE_IDS), "; context_budget =", CONTEXT_BUDGET)

model = glm-4-flash ; cases = 4 ; context_budget = 1800


## 1. 检索：只接收问题文字和 canonical evidence

从 canonical 问题文件投影出 `query_id/text`，从证据文件投影出 `evidence_id/page/quote`。检索器不读取训练候选或评测字段。“模型评估与宏/微平均”题的首轮只取 BM25 第一条，显式模拟召回范围较窄的应用设置；普通题仍取前八条。

In [2]:
# 检索前的输入边界：明确只构造 query_id/text 投影；不调用会加载
# C2 候选的完整数据集读取器。
query_rows = read_jsonl(DATASET_ROOT / "queries.jsonl")
query_only = {
    str(row["query_id"]): str(row["text"])
    for row in query_rows
    if row.get("query_id") in CASE_IDS
}
if set(query_only) != set(CASE_IDS):
    raise AssertionError("固定 capstone 问题缺失")

all_evidence = read_jsonl(DATASET_ROOT / "evidence.jsonl")
canonical_evidence = [
    {
        "evidence_id": str(row["evidence_id"]),
        "page": int(row["page"]),
        "quote": str(row["quote"]),
    }
    for row in all_evidence
    if row.get("evidence_type") == "quote"
]
if not canonical_evidence:
    raise ValueError("canonical quote evidence 为空")
if any(row["evidence_id"].startswith("evi_ft_") for row in canonical_evidence):
    raise AssertionError("C2 训练候选混入 capstone evidence 池")

search = build_bm25_chunk_search(
    [
        {"chunk_id": row["evidence_id"], "pages": [row["page"]], "text": row["quote"]}
        for row in canonical_evidence
    ]
)
initial_top_k = {case_id: (1 if case_id == GAP_CASE_ID else TOP_K) for case_id in CASE_IDS}
retrieved = {
    case_id: search(query, top_k=initial_top_k[case_id])
    for case_id, query in query_only.items()
}
if any(not rows for rows in retrieved.values()):
    raise AssertionError("至少有一道题没有检索结果")

retrieved_ids = {
    case_id: [item.chunk_id for item in rows]
    for case_id, rows in retrieved.items()
}
initial_retrieved_ids = {case_id: list(ids) for case_id, ids in retrieved_ids.items()}
for case_id in CASE_IDS:
    print(f"\n[{case_id}] {query_only[case_id]}")
    for rank, item in enumerate(retrieved[case_id], start=1):
        print(f"  {rank:>2}. {item.chunk_id} | p.{item.pages[0]} | score={item.score:.3f}")

print("\n检索阶段完成：排序输入只有 query text 与 canonical quote；qrels 尚未读取。")


[model_selection_with_intro_scope] 《南瓜书》绪论里说，机器学习算法之间有没有绝对更好的一个？
   1. evi_065c349a8cbf | p.17 | score=38.044
   2. evi_4534a9518d9a | p.15 | score=24.274
   3. evi_718a2d52ed5c | p.18 | score=23.856
   4. evi_451af37f42ba | p.118 | score=22.086
   5. evi_32c0b230bb7c | p.17 | score=20.795
   6. evi_9b2016615541 | p.88 | score=20.298
   7. evi_8845538704bf | p.16 | score=19.029
   8. evi_3bc6b53ddcaf | p.100 | score=16.856

[cross_validation_reliability] 交叉验证法为什么比单次留出法更可靠？
   1. evi_718a2d52ed5c | p.18 | score=27.792
   2. evi_df934cb5bae0 | p.18 | score=27.755
   3. evi_9a292ab3d933 | p.19 | score=26.416
   4. evi_5f5d3dc01c2d | p.19 | score=26.135
   5. evi_833c8fa10a30 | p.19 | score=26.049
   6. evi_4c24466586af | p.18 | score=22.653
   7. evi_40e63a429c30 | p.19 | score=16.742
   8. evi_24f3ee133e62 | p.39 | score=10.452

[model_evaluation_and_macro_micro] “模型评估与选择”是什么？为什么类别不平衡时要区分宏平均和微平均？
   1. evi_91d0b1e7d0ff | p.18 | score=33.879

[book_evidence_boundary] 《南瓜书》建议使用哪个 CUDA 版本训

## 2. 上下文预算：按完整证据逐条截断

上下文预算不是把 quote 从中间切断，而是按 BM25 顺序加入完整 evidence。这样后续引用可以用 `evidence_id` 找回 canonical 原文；若第一条证据本身超过预算，直接失败而不静默截断。

In [3]:
evidence_by_id = {row["evidence_id"]: row for row in canonical_evidence}

def context_block(row):
    return f"[evidence_id={row['evidence_id']} page={row['page']}]\n{row['quote']}"

def apply_context_budget(items, budget=CONTEXT_BUDGET):
    separator = "\n\n"
    selected = []
    used = 0
    for item in items:
        row = evidence_by_id[item.chunk_id]
        block = context_block(row)
        if len(block) > budget and not selected:
            raise ValueError(f"首条 evidence 超过上下文预算：{item.chunk_id}")
        added = len(block) + (len(separator) if selected else 0)
        if used + added > budget:
            break
        selected.append(row)
        used += added
    if not selected:
        raise ValueError("上下文预算后没有可用 evidence")
    return selected, used

context_rows = {}
context_text = {}
for case_id, items in retrieved.items():
    rows, used = apply_context_budget(items)
    context_rows[case_id] = rows
    context_text[case_id] = "\n\n".join(context_block(row) for row in rows)
    print(f"{case_id}: {len(rows)} 条 evidence，{used}/{CONTEXT_BUDGET} chars")
    if len(context_text[case_id]) != used:
        raise AssertionError("上下文预算统计与实际 join 长度不一致")
    if len(context_text[case_id]) > CONTEXT_BUDGET:
        raise AssertionError("实际拼接后的上下文超过字符预算")
    if not set(retrieved_ids[case_id]) >= {row["evidence_id"] for row in rows}:
        raise AssertionError("上下文 evidence 不在检索结果中")

# 保留首轮输入，补查后仍能重放完全相同的预算。
initial_context_rows = {case_id: list(rows) for case_id, rows in context_rows.items()}
initial_context_text = dict(context_text)

cross_validation_reliability: 8 条 evidence，1251/1800 chars
book_evidence_boundary: 8 条 evidence，1637/1800 chars
model_evaluation_and_macro_micro: 1 条 evidence，79/1800 chars
model_selection_with_intro_scope: 7 条 evidence，1554/1800 chars


## 3. 接入 C6：检查首轮缺口，按模型原始补查词重检索

本题仍是完整的用户问题，首轮只返回 BM25 第一条。模型按 C6 的 `sufficient/missing/repair_queries/supporting_quotes` 四字段契约检查资料；任何支持引文都必须绑定到首轮实际 quote。只有真实响应 `sufficient=false` 才执行它给出的补查词，人工不补入答案语义。

补查结果按“首轮在前、补查排名在后”稳定去重，再用完全相同的 1,800 字符预算选择完整 evidence。本单元保存原始反思、触发理由、每次检索词和检索结果；最后再用 qrels 检查是否确实补到了先前缺失的证据。

In [4]:
gap_repairs = {}
gap_question = query_only[GAP_CASE_ID]
gap_prompt = (
    "你是资料问答流程中的严格缺口检查器。只根据 QUESTION 和 INITIAL_CONTEXT 判断每个独立要求是否都有直接证据。"
    "先在心中按问号、并列连词和‘是什么/为什么/如何’拆出所有要求，再逐项寻找直接证据；只要一项没有直接证据就必须 sufficient=false。"
    "不要凭主题相近、只覆盖其中一问或外部常识宣布资料足够。只输出严格 JSON，字段必须精确为 "
    "sufficient、missing、repair_queries、supporting_quotes。"
    "sufficient 必须是 boolean；另外三项必须是无重复字符串列表。"
    "资料足够时 sufficient=true，missing=[]、repair_queries=[]，supporting_quotes 至少一条候选原文。"
    "资料不足时 sufficient=false，missing 必须写出缺少的要求，repair_queries 给 1 至 3 条实际可搜索的补查词；"
    "supporting_quotes 可以为空，其非空项必须是 INITIAL_CONTEXT 某一条 quote 中的逐字短语，不能从问题中编引文。"
    "补查词只使用 QUESTION 或 INITIAL_CONTEXT 可见的概念，不把尚未检索到的结论写成事实，不能虚构 evidence_id。"
    "只输出 JSON，不要 Markdown 围栏。\n\n"
    f"QUESTION:\n{gap_question}\n\nINITIAL_CONTEXT:\n{initial_context_text[GAP_CASE_ID]}"
)
gap_raw = llm_call(gap_prompt, max_tokens=600)
persist_raw_response("gap_reflection", GAP_CASE_ID, gap_raw)
gap_reflection = validate_gap_reflection(gap_raw, initial_context_rows[GAP_CASE_ID])
gap_triggered = not gap_reflection["sufficient"]
repair_trace = {
    "raw_reflection": gap_raw,
    "reflection": gap_reflection,
    "triggered": gap_triggered,
    "trigger_reason": (
        "reflection.sufficient=false: " + "；".join(gap_reflection["missing"])
        if gap_triggered else "reflection.sufficient=true"
    ),
    "retrieval_rounds": [],
}
if gap_triggered:
    merged = list(retrieved[GAP_CASE_ID])
    seen = {item.chunk_id for item in merged}
    for repair_query in gap_reflection["repair_queries"]:
        follow_hits = search(repair_query, top_k=TOP_K)
        repair_trace["retrieval_rounds"].append({
            "query": repair_query,
            "retrieved_ids": [item.chunk_id for item in follow_hits],
        })
        for item in follow_hits:
            if item.chunk_id not in seen:
                seen.add(item.chunk_id)
                merged.append(item)
    retrieved[GAP_CASE_ID] = merged
    retrieved_ids[GAP_CASE_ID] = [item.chunk_id for item in merged]
    rows, used = apply_context_budget(merged, budget=CONTEXT_BUDGET)
    context_rows[GAP_CASE_ID] = rows
    context_text[GAP_CASE_ID] = "\n\n".join(context_block(row) for row in rows)
    if len(context_text[GAP_CASE_ID]) != used or used > CONTEXT_BUDGET:
        raise AssertionError("补查后的实际上下文不符合相同字符预算")
gap_repairs[GAP_CASE_ID] = repair_trace
print("首轮缺口判断：", gap_reflection)
print("是否触发补查：", gap_triggered, "；理由：", repair_trace["trigger_reason"])
for round_ in repair_trace["retrieval_rounds"]:
    print("实际补查：", round_["query"], "→", round_["retrieved_ids"])
print("前后上下文字符：", len(initial_context_text[GAP_CASE_ID]), "→", len(context_text[GAP_CASE_ID]), "/", CONTEXT_BUDGET)


首轮缺口判断： {'sufficient': False, 'missing': ['模型评估与选择是什么', '为什么类别不平衡时要区分宏平均和微平均'], 'repair_queries': ['模型评估与选择定义', '宏平均和微平均类别不平衡原因'], 'supporting_quotes': []}
是否触发补查： True ；理由： reflection.sufficient=false: 模型评估与选择是什么；为什么类别不平衡时要区分宏平均和微平均
实际补查： 模型评估与选择定义 → ['evi_91d0b1e7d0ff', 'evi_2b63acecd8f5', 'evi_df934cb5bae0', 'evi_718a2d52ed5c', 'evi_e6b0856aad44', 'evi_e90514a46bf2', 'evi_aa9d4ed41417', 'evi_ac1c32543e00']
实际补查： 宏平均和微平均类别不平衡原因 → ['evi_ffd64ad64243', 'evi_e99545deedf4', 'evi_9b9343451859', 'evi_09c268bb07e9', 'evi_cf744a5b9df8', 'evi_243ea8f28452', 'evi_e99532b3bdc7', 'evi_c6dc9d6de656']
前后上下文字符： 79 → 1640 / 1800


## 4. 真实回答：模型只返回身份引用，quote 由 Notebook hydrate

四道题统一调用一次真实 `glm-4-flash`，模型根据本题最终 QUESTION+CONTEXT 返回 `answerable` 或 `insufficient`。引用只允许来自本次上下文的 `evidence_id`，quote 从 canonical 原文逐字 hydrate。`check_capstone.py` 提供可独立测试的严格 JSON 解析器：通过 `json.loads` 拒绝多余字段、重复字段、额外文字和伪造引用。

In [5]:
def generation_prompt(question, context):
    return (
        "你是一个严格的资料问答器。只能依据 CONTEXT 回答 QUESTION，不得使用外部知识。status 只能是 answerable 或 insufficient。\n"
        "请只输出一个 JSON 对象，不要 Markdown 代码围栏，格式必须是："
        '{"status":"answerable","answer":"...","citations":[{"evidence_id":"..."}]}\n'
        "如果 CONTEXT 不足以回答，必须返回 status=insufficient、answer=“资料不足，无法根据提供的上下文回答。”、citations=[]，不得添加其他内容。"
        "answerable 时 citations 必须列出支持答案的全部 evidence_id；只能从 CONTEXT 复制身份，"
        "不要输出 quote 字段，Notebook 会按 evidence_id 从 canonical evidence 逐字 hydrate。\n\n"
        f"QUESTION:\n{question}\n\nCONTEXT:\n{context}"
    )

def validate_model_json(raw, allowed_ids):
    # 包含 INSUFFICIENT_ANSWER 固定拒答句检查；无 JSON 修补或默认答案。
    return validate_generation(raw, allowed_ids)

raw_model = {}
responses = {}
for case_id in CASE_IDS:
    raw = llm_call(
        generation_prompt(query_only[case_id], context_text[case_id]),
        max_tokens=700,
    )
    raw_model[case_id] = raw
    persist_raw_response("generation", case_id, raw)
    parsed = validate_model_json(raw, {row["evidence_id"] for row in context_rows[case_id]})
    hydrated = [
        {
            "evidence_id": evidence_id,
            "page": evidence_by_id[evidence_id]["page"],
            "quote": evidence_by_id[evidence_id]["quote"],
        }
        for evidence_id in parsed["citation_ids"]
    ]
    responses[case_id] = {
        "status": parsed["status"],
        "answer": parsed["answer"],
        "citation_ids": parsed["citation_ids"],
        "hydrated_citations": hydrated,
        "model_called": True,
    }
    print(f"{case_id}: glm-4-flash 返回 {parsed['status']}，{len(hydrated)} 条引用")

model_selection_with_intro_scope: glm-4-flash 返回 answerable，2 条引用


cross_validation_reliability: glm-4-flash 返回 answerable，7 条引用


model_evaluation_and_macro_micro: glm-4-flash 返回 answerable，16 条引用


book_evidence_boundary: glm-4-flash 返回 insufficient，0 条引用


## 5. 独立请求语义检查（同一 `glm-4-flash`）：逐条判断结论是否被 quote 蕴含

合法引用 ID 不能证明答案正确。这里再次调用真实 `glm-4-flash`，只把问题、回答状态、回答文字和已 hydrate 的 quote 提交给 reviewer，逐条返回 `supported/contradicted/not_found`。存在未被支持的主要结论或 reviewer 判为 `fail`，会得到质量负结果并保留其原始响应。

评审是另一次独立请求，但仍使用与生成相同的模型，可能存在共同偏差；这里的“独立”不是独立模型，也不是人工复核。四道固定题上的模型辅助审计不能代替人工事实证明。

In [6]:
SEMANTIC_JUDGE_PROMPT = (
    "你是独立的回答语义审计器，不是回答者。只能依据 QUESTION、ANSWER 和 CITED_EVIDENCE 判断，不能使用外部知识。\n"
    "把 ANSWER 拆成不超过 5 条主要可验证结论。每条严格返回 claim、relation、evidence_ids；"
    "relation 只能是 supported（对应 quote 直接蕴含结论）、contradicted（对应 quote 明确反驳）或 not_found（引用 quote 既未蕴含也未反驳）。\n"
    "supported/contradicted 必须列出实际支持或反驳该结论的 evidence_id，not_found 的 evidence_ids 必须为空。"
    "只有所有主要结论都 supported 且确实回应 QUESTION 时 verdict 才能是 pass；否则必须是 fail。\n"
    "若 STATUS=insufficient，只有 ANSWER 恰好是资料不足声明时才允许 claims=[]、verdict=pass；不能夹带任何答案或推测。\n"
    "只输出严格 JSON：{{\"claims\":[{{\"claim\":\"...\",\"relation\":\"supported|contradicted|not_found\",\"evidence_ids\":[\"...\"]}}],\"verdict\":\"pass|fail\",\"reason\":\"...\"}}。\n\n"
    "QUESTION: {question}\nSTATUS: {status}\nANSWER: {answer}\nCITED_EVIDENCE:\n{evidence}"
)

def validate_semantic_judge(raw, allowed_ids, generation_status):
    return validate_semantic_review(raw, allowed_ids, generation_status)

semantic_results = {}
raw_semantic = {}
for case_id in CASE_IDS:
    response = responses[case_id]
    cited_evidence = "\n\n".join(
        context_block(row) for row in response.get("hydrated_citations", [])
    ) or "(none)"
    raw = llm_call(
        SEMANTIC_JUDGE_PROMPT.format(
            question=query_only[case_id],
            status=response["status"],
            answer=response["answer"],
            evidence=cited_evidence,
        ),
        max_tokens=900,
    )
    raw_semantic[case_id] = raw
    persist_raw_response("semantic_review", case_id, raw)
    parsed = validate_semantic_judge(
        raw, set(response.get("citation_ids", [])), response["status"]
    )
    semantic_results[case_id] = {
        "claims": parsed["claims"],
        "verdict": parsed["verdict"],
        "reason": parsed["reason"],
        "model_called": True,
    }
    print(f"{case_id}: semantic judge glm-4-flash → {parsed['verdict']}，{len(parsed['claims'])} 条 claim")
    for claim in parsed["claims"]:
        print(f"  {claim['relation']}: {claim['claim']} | evidence={claim['evidence_ids']}")

print("语义检查限制：这是独立模型辅助审计，只在固定 4 题和已 hydrate quote 上工作，不能替代人工事实证明或外推到一般任务。")

model_selection_with_intro_scope: semantic judge glm-4-flash → pass，2 条 claim
  supported: 机器学习算法之间没有绝对的优劣之分 | evidence=['evi_065c349a8cbf', 'evi_32c0b230bb7c']
  supported: 机器学习算法的优劣取决于是否适合当前待解决的问题 | evidence=['evi_065c349a8cbf', 'evi_32c0b230bb7c']


cross_validation_reliability: semantic judge glm-4-flash → pass，5 条 claim
  supported: 交叉验证法比单次留出法更可靠。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 交叉验证法本质上是在进行多次留出法。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 交叉验证法每次都换不同的子集做测试集。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 所有样本在交叉验证法中至少做1次测试样本。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 单次留出法仅依靠1组训练集和测试集对比不同算法效果不够置信，偶然性太强。 | evidence=['evi_833c8fa10a30', 'evi_40e63a429c30']


model_evaluation_and_macro_micro: semantic judge glm-4-flash → pass，5 条 claim
  supported: 模型评估与选择是评估模型优劣和选择适合业务场景模型的过程。 | evidence=['evi_91d0b1e7d0ff', 'evi_2b63acecd8f5']
  supported: 宏平均在类别不平衡时平等看待每个类别，会受到高P和高R类别的影响。 | evidence=['evi_ffd64ad64243']
  supported: 微平均考虑了每个类别的样本数量，样本数量多的类别会主导最终结果。 | evidence=['evi_ffd64ad64243', 'evi_e99532b3bdc7']
  supported: 宏平均没有考虑每个类别下的样本数量。 | evidence=['evi_ffd64ad64243', 'evi_c6dc9d6de656']
  supported: 微平均则考虑到了每个类别的样本数量。 | evidence=['evi_ffd64ad64243', 'evi_e99532b3bdc7']


book_evidence_boundary: semantic judge glm-4-flash → pass，0 条 claim
语义检查限制：这是独立模型辅助审计，只在固定 4 题和已 hydrate quote 上工作，不能替代人工事实证明或外推到一般任务。


## 6. 事后验收：缺口变化、qrels 覆盖、逐字引用和语义拒答

所有模型调用结束后才读取评测标注。answerable 题逐项检查 essential evidence 是否被检索、进入预算并被引用，以及 reviewer 是否认为主要结论均有 quote 支持；unanswerable 题检查固定拒答句与空引用。

信息缺口题额外比较首轮缺失的必要证据和补查加入的新证据，最终回答通过时才记录 `gap_closed=true`。缺口未触发、检索没有改善或回答不合格都保留真实结果，不把它们改成通过。

In [7]:
# 生成和 reviewer 完成后才读取评测标签；不会回流模型。
query_meta = {
    row["query_id"]: row
    for row in read_jsonl(DATASET_ROOT / "queries.jsonl")
    if row["query_id"] in CASE_IDS
}
if set(query_meta) != set(CASE_IDS):
    raise AssertionError("canonical query metadata 不完整")
if any("regression" not in row.get("usage", []) for row in query_meta.values()):
    raise AssertionError("capstone 只能使用 canonical regression 问题")

qrels_by_case = {}
for row in read_jsonl(DATASET_ROOT / "qrels.jsonl"):
    if row["query_id"] in CASE_IDS:
        qrels_by_case.setdefault(row["query_id"], []).append(row)

audit_rows = []
for case_id in CASE_IDS:
    metadata = query_meta[case_id]
    qrels = qrels_by_case.get(case_id, [])
    positives = {row["evidence_id"] for row in qrels if int(row["relevance"]) == 1}
    essential = {
        row["evidence_id"] for row in qrels
        if int(row["relevance"]) == 1 and bool(row.get("essential"))
    }
    response = responses[case_id]
    semantic = semantic_results[case_id]
    row = {
        "query_id": case_id,
        "question": query_only[case_id],
        "answerability": metadata["answerability"],
        "initial_top_k": initial_top_k[case_id],
        "initial_retrieved_ids": initial_retrieved_ids[case_id],
        "initial_context_evidence_ids": [r["evidence_id"] for r in initial_context_rows[case_id]],
        "initial_context_chars": len(initial_context_text[case_id]),
        "retrieved_ids": retrieved_ids[case_id],
        "context_evidence_ids": [r["evidence_id"] for r in context_rows[case_id]],
        "context_chars": len(context_text[case_id]),
        "citation_ids": response["citation_ids"],
        "hydrated_citations": response["hydrated_citations"],
        "qrels_positive_citation_ids": sorted(set(response["citation_ids"]) & positives),
        "essential_ids": sorted(essential),
        "raw_generation_response": raw_model[case_id],
        "response_status": response["status"],
        "answer": response["answer"],
        "model_called": response["model_called"],
        "raw_semantic_response": raw_semantic[case_id],
        "semantic_verdict": semantic["verdict"],
        "semantic_claims": semantic["claims"],
        "semantic_reason": semantic["reason"],
        "semantic_judge_model_called": semantic["model_called"],
        "gap_repair": gap_repairs.get(case_id),
    }
    row["verdict"], row["failure_reasons"] = evaluate_case(row, metadata, qrels)
    if row["gap_repair"] is not None:
        repair = row["gap_repair"]
        first_ids = set(row["initial_context_evidence_ids"])
        new_evidence = [r for r in context_rows[case_id] if r["evidence_id"] not in first_ids]
        missing_before = sorted(essential - first_ids)
        repair["first_round_missing_essential_ids"] = missing_before
        repair["new_evidence"] = new_evidence
        repair["gap_closed"] = bool(
            repair["triggered"] and missing_before and new_evidence
            and essential <= set(row["context_evidence_ids"])
            and row["verdict"] == "pass_answerable"
        )
    audit_rows.append(row)

for row in audit_rows:
    print(f"\n[{row['query_id']}] {row['response_status']}")
    print(row["answer"])
    for citation in row["hydrated_citations"]:
        print(f"  cite {citation['evidence_id']} | p.{citation['page']} | quote={citation['quote']}")
    if row["gap_repair"] is not None:
        repair = row["gap_repair"]
        print("首轮事后缺口：", repair["first_round_missing_essential_ids"])
        print("补查触发理由：", repair["trigger_reason"])
        print("新上下文证据：", [r["evidence_id"] for r in repair["new_evidence"]])
        print("同一预算：", row["initial_context_chars"], "→", row["context_chars"], "/", CONTEXT_BUDGET)
        print("信息缺口闭环通过：", repair["gap_closed"])
    print("质量验收：", row["verdict"], row["failure_reasons"])

all_passed = (
    all(row["verdict"].startswith("pass_") for row in audit_rows)
    and gap_repairs[GAP_CASE_ID]["gap_closed"]
)
capstone_audit = {
    "schema_version": 2,
    "trace_kind": "c7_endpoint_capstone",
    "model": RAG_LLM_MODEL,
    "api_key_source": ".env:ZHIPUAI_API_KEY",
    "max_retries": 0,
    "retriever": "bm25",
    "evidence_scope": "canonical evidence_type=quote only",
    "top_k": TOP_K,
    "context_budget_chars": CONTEXT_BUDGET,
    "gap_case_id": GAP_CASE_ID,
    "semantic_judge": {
        "model": RAG_LLM_MODEL,
        "calls": len(CASE_IDS),
        "limitation": "独立请求使用同一个 glm 模型，可能有共同偏差；只覆盖固定 4 题，不能替代人工事实证明。",
    },
    "cases": audit_rows,
    "all_passed": all_passed,
}
emit_tutorial_audit(capstone_audit)
print("\n全部质量验收通过：", all_passed)


[model_selection_with_intro_scope] answerable
机器学习算法之间没有绝对的优劣之分，只有是否适合当前待解决的问题之分
  cite evi_065c349a8cbf | p.17 | quote=机器学习算法之间没有绝对的优劣之分，只有是否适合当前待解决的问题之分
  cite evi_32c0b230bb7c | p.17 | quote=没有绝对的优劣之分，只有是否适合当前待解决的问题之分
质量验收： pass_answerable []

[cross_validation_reliability] answerable
交叉验证法比单次留出法更可靠的原因在于交叉验证法本质上是在进行多次留出法，且每次都换不同的子集做测试集，最终让所有样本均至少做1次测试样本。这样做的理由是因为一般的留出法只会划分出1组训练集和测试集，仅依靠1组训练集和测试集去对比不同算法之间的效果显然不够置信，偶然性太强。通过多次划分，每次采用不同的子集作为测试集，可以减少偶然性，提高评估结果的可靠性。
  cite evi_718a2d52ed5c | p.18 | quote=2.2 评估方法 本节介绍了3 种模型评估方法：留出法、交叉验证法、自助法。留出法由于操作简单，因此最常用； 交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果；自助法常用 于集成学习（详见“西瓜书”第8 章的8.2 节和8.3 节）产生基分类器。留出法和自助法简单易懂，在此 不再赘述，下面举例说明交叉验证法的常用方式。 对比同一算法的不同参数配置之间的效果：假设现有数据集D，且有一个被评估认为适合用于数据集 D 的算法L，该算法有可配置的参数，假设备选的参数配置方案有两套：方案a，方案b。下
  cite evi_df934cb5bae0 | p.18 | quote=本节介绍了3 种模型评估方法：留出法、交叉验证法、自助法
  cite evi_9a292ab3d933 | p.19 | quote=从以上的举例可以看出，交叉验证法本质上是在进行多次留出法，且每次都换不同的子集做测试集， 最终让所有样本均至少做1 次测试样本。
  cite evi_5f5d3dc01c2d | p.19 | quote=交叉验证法本质上是在进行多次留出


全部质量验收通过： True


这个 capstone 保存一条可检查的运行链：首轮检索与预算、模型缺口判断、实际补查、新证据、最终回答和同一模型的独立请求语义 reviewer。窄召回样例说明 C6 控制流如何接入最终验收，不能用来推断补查在其他问题上一定有收益。

从 C7 根目录运行 `python '7. 评估/check_capstone.py'` 会重放本地检索和预算、重新解析所有原始响应，并核对 canonical quote/qrels。默认检查审计是否真实一致，允许如实保存的质量负结果；加上 `--require-all-passed` 才要求所有回答与补查闭环通过。该检查不能证明第三方 API 来源的密码学真实性，也不能代替人工判断 reviewer 的语义结论。

## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[持续观察结果](持续观察教程和线上结果.ipynb)

